# Smart Waste – Training Klasifikasi Gambar Sampah

Model klasifikasi 6 kelas (**glass, metal, organic, paper, plastic, textile**) dengan *transfer learning* MobileNetV2.

**Sebelum mulai:**
1. Upload `dataset_6class.zip` ke Google Drive di `MyDrive/smart-waste/` (hasil `scripts/prepare_dataset.py`).
2. Di Colab: **Runtime → Change runtime type → T4 GPU**.
3. Jalankan semua sel (**Runtime → Run all**).

Hasil (model, label, grafik, dan metrik evaluasi) disimpan ke `MyDrive/smart-waste/outputs/`.

## 1. Setup

In [ ]:
# Cek GPU
!nvidia-smi -L

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/smart-waste"
ZIP_PATH = f"{DRIVE_DIR}/dataset_6class.zip"
DATA_DIR = "/content/data"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"

!mkdir -p "{DATA_DIR}" "{OUTPUT_DIR}"
!unzip -q -o "{ZIP_PATH}" -d "{DATA_DIR}"
!ls "{DATA_DIR}"

In [ ]:
import json
import os

import keras
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras import layers
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Load Dataset

In [ ]:
def load_split(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, split),
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="int",
        shuffle=shuffle,
        seed=SEED,
    )


train_ds = load_split("train", shuffle=True)
val_ds = load_split("val", shuffle=False)
test_ds = load_split("test", shuffle=False)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
test_file_paths = test_ds.file_paths
print("Kelas:", CLASS_NAMES)

In [ ]:
# Contoh gambar per batch
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 3. Model

Input berupa gambar RGB 224×224 dengan nilai piksel 0–255. Augmentasi dan normalisasi ada di dalam model, jadi API prediksi cukup mengirim gambar yang sudah di-resize tanpa preprocessing tambahan. Augmentasi hanya aktif saat training.

In [ ]:
augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
        layers.RandomBrightness(0.1),
    ],
    name="augmentation",
)

base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)  # sama dengan mobilenet_v2.preprocess_input
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="waste_classifier")
model.summary()

## 4. Training Tahap 1 – Classifier Head

In [ ]:
EPOCHS_HEAD = 10

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks)

## 5. Training Tahap 2 – Fine-tuning

Membuka 30 layer terakhir MobileNetV2 dengan learning rate kecil. Layer BatchNormalization tetap dalam mode inferensi karena `base_model` dipanggil dengan `training=False`.

In [ ]:
EPOCHS_FINE = 10
FINE_TUNE_LAYERS = 30

base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

initial_epoch = len(history_head.history["loss"])
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epoch + EPOCHS_FINE,
    initial_epoch=initial_epoch,
    callbacks=callbacks,
)

In [ ]:
# Grafik akurasi dan loss (garis putus-putus = mulai fine-tuning)
hist = {k: history_head.history[k] + history_fine.history[k] for k in history_head.history}
epochs = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in [(axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")]:
    ax.plot(epochs, hist[metric], label="train")
    ax.plot(epochs, hist[f"val_{metric}"], label="validation")
    ax.axvline(initial_epoch + 0.5, color="gray", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_history.png", dpi=150)
plt.show()

## 6. Evaluasi pada Data Test

In [ ]:
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
print(f"Test accuracy: {test_accuracy:.4f}\n")

report_text = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
print(report_text)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title(f"Confusion Matrix (test accuracy {test_accuracy:.2%})")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Akurasi terpisah untuk foto studio (default) dan foto dunia nyata (real_world)
variants = np.array(["real_world" if "__real_world__" in p else "default" for p in test_file_paths])
accuracy_by_variant = {}
for v in ["default", "real_world"]:
    mask = variants == v
    accuracy_by_variant[v] = float(accuracy_score(y_true[mask], y_pred[mask]))
    print(f"{v:<11} n={mask.sum():<4} accuracy={accuracy_by_variant[v]:.4f}")

In [ ]:
# Contoh prediksi yang salah
wrong = np.where(y_true != y_pred)[0]
print(f"{len(wrong)} dari {len(y_true)} gambar salah diprediksi")

plt.figure(figsize=(14, 7))
for i, idx in enumerate(wrong[:8]):
    img = keras.utils.load_img(test_file_paths[idx])
    ax = plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(f"aktual: {CLASS_NAMES[y_true[idx]]}\nprediksi: {CLASS_NAMES[y_pred[idx]]} ({y_prob[idx].max():.0%})", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Simpan Model dan Hasil Evaluasi

In [ ]:
model.save(f"{OUTPUT_DIR}/waste_classifier.keras")

with open(f"{OUTPUT_DIR}/labels.json", "w") as f:
    json.dump(CLASS_NAMES, f)

with open(f"{OUTPUT_DIR}/classification_report.txt", "w") as f:
    f.write(f"Test accuracy: {test_accuracy:.4f}\n\n{report_text}")

metrics = {
    "test_accuracy": float(test_accuracy),
    "accuracy_by_variant": accuracy_by_variant,
    "classification_report": report_dict,
    "confusion_matrix": cm.tolist(),
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "epochs_trained": len(hist["loss"]),
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

!ls -lh "{OUTPUT_DIR}"

## 8. Uji Prediksi Satu Gambar

In [ ]:
loaded = keras.models.load_model(f"{OUTPUT_DIR}/waste_classifier.keras")


def predict_image(path):
    img = keras.utils.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    x = np.expand_dims(keras.utils.img_to_array(img), 0)
    prob = loaded.predict(x, verbose=0)[0]
    return CLASS_NAMES[int(prob.argmax())], float(prob.max())


sample = test_file_paths[0]
print(sample)
print(predict_image(sample))